# Chapter 7 — Object-Oriented Design

Approach to OOD questions:
1. **Handle ambiguity**: ask who, what, where, when, how, and why.
2. **Define the core objects.**
3. **Analyze relationships** (has-a, is-a, one-to-many…).
4. **Investigate actions** (what does each object do?).

Useful patterns: **Singleton**, **Factory Method**, and others (Strategy, Observer...).

Modern Python tools used here: `dataclass`, `Enum`, `abc.ABC`, `typing.Protocol`, and `match`.

In [1]:
from __future__ import annotations

import random
import threading
from abc import ABC, abstractmethod
from collections import deque
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum, IntEnum, auto
from typing import ClassVar, Generic, Iterator, TypeVar

T = TypeVar("T")


class Singleton:
    """Thread-safe singleton via __new__."""
    _instance: ClassVar[Singleton | None] = None
    _lock: ClassVar[threading.Lock] = threading.Lock()

    def __new__(cls) -> Singleton:
        with cls._lock:
            if cls._instance is None:
                cls._instance = super().__new__(cls)
        return cls._instance


class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

    @staticmethod
    def factory(kind: str, *args: float) -> Shape:  # Factory Method
        match kind:
            case "square": return Square(*args)
            case "circle": return Circle(*args)
        raise ValueError(kind)


@dataclass
class Square(Shape):
    side: float
    def area(self) -> float: return self.side ** 2


@dataclass
class Circle(Shape):
    r: float
    def area(self) -> float: return 3.14159 * self.r ** 2


assert Singleton() is Singleton()
assert Shape.factory("square", 3).area() == 9
print("patterns ok")

patterns ok


## 7.1 Deck of Cards
Design a generic deck of cards, then show how to subclass it for Blackjack.

In [2]:
class Suit(Enum):
    CLUBS = "♣"; DIAMONDS = "♦"; HEARTS = "♥"; SPADES = "♠"


@dataclass(frozen=True)
class Card:
    rank: int  # 1 = Ace ... 11 = J, 12 = Q, 13 = K
    suit: Suit

    def __str__(self) -> str:
        names = {1: "A", 11: "J", 12: "Q", 13: "K"}
        return f"{names.get(self.rank, self.rank)}{self.suit.value}"


C = TypeVar("C", bound=Card)


class Deck(Generic[C]):
    def __init__(self, cards: list[C]) -> None:
        self._cards = cards
        self._dealt = 0

    def shuffle(self) -> None:
        random.shuffle(self._cards)
        self._dealt = 0

    def remaining(self) -> int:
        return len(self._cards) - self._dealt

    def deal(self, n: int = 1) -> list[C]:
        if n > self.remaining():
            raise ValueError("not enough cards")
        hand = self._cards[self._dealt:self._dealt + n]
        self._dealt += n
        return hand


class Hand(Generic[C]):
    def __init__(self) -> None:
        self.cards: list[C] = []

    def add(self, *cards: C) -> None:
        self.cards.extend(cards)

    def score(self) -> int:
        return sum(c.rank for c in self.cards)


class BlackjackCard(Card):
    def min_value(self) -> int:
        return 1 if self.rank == 1 else min(self.rank, 10)


class BlackjackHand(Hand[BlackjackCard]):
    def score(self) -> int:
        total = sum(c.min_value() for c in self.cards)
        aces = sum(c.rank == 1 for c in self.cards)
        for _ in range(aces):  # upgrade aces from 1 to 11 while we don't bust
            if total + 10 <= 21:
                total += 10
        return total

    def busted(self) -> bool: return self.score() > 21
    def is_blackjack(self) -> bool: return len(self.cards) == 2 and self.score() == 21


deck = Deck([BlackjackCard(r, s) for s in Suit for r in range(1, 14)])
deck.shuffle()
hand = BlackjackHand()
hand.add(*deck.deal(2))
print("hand:", " ".join(map(str, hand.cards)), "score:", hand.score())
h = BlackjackHand(); h.add(BlackjackCard(1, Suit.SPADES), BlackjackCard(13, Suit.HEARTS))
assert h.is_blackjack()
h.add(BlackjackCard(1, Suit.CLUBS)); assert h.score() == 12

hand: 5♠ K♣ score: 15


## 7.2 Call Center
Three employee levels: respondent → manager → director. An incoming call goes to a free respondent. If a respondent can't handle it, it escalates to a manager, and from there to a director. Implement `dispatch_call()`.

In [3]:
class Rank(IntEnum):
    RESPONDENT = 0
    MANAGER = 1
    DIRECTOR = 2


@dataclass(eq=False)
class Call:
    caller: str
    rank: Rank = Rank.RESPONDENT
    handler: Employee | None = None

    def escalate(self) -> None:
        self.rank = Rank(min(self.rank + 1, Rank.DIRECTOR))


@dataclass(eq=False)
class Employee:
    name: str
    rank: Rank
    center: CallCenter
    current: Call | None = None

    def is_free(self) -> bool:
        return self.current is None

    def receive(self, call: Call) -> None:
        self.current, call.handler = call, self

    def complete(self) -> None:
        self.current = None
        self.center.assign_waiting(self)

    def escalate(self) -> None:
        call, self.current = self.current, None
        assert call is not None
        call.escalate()
        self.center.dispatch_call(call)
        self.center.assign_waiting(self)


class CallCenter:
    def __init__(self, respondents: int, managers: int, directors: int) -> None:
        self.staff: dict[Rank, list[Employee]] = {
            Rank.RESPONDENT: [Employee(f"R{i}", Rank.RESPONDENT, self) for i in range(respondents)],
            Rank.MANAGER: [Employee(f"M{i}", Rank.MANAGER, self) for i in range(managers)],
            Rank.DIRECTOR: [Employee(f"D{i}", Rank.DIRECTOR, self) for i in range(directors)],
        }
        self.queues: dict[Rank, deque[Call]] = {r: deque() for r in Rank}

    def _free_handler(self, call: Call) -> Employee | None:
        for rank in Rank:
            if rank >= call.rank:
                for e in self.staff[rank]:
                    if e.is_free():
                        return e
        return None

    def dispatch_call(self, call: Call) -> Employee | None:
        e = self._free_handler(call)
        if e:
            e.receive(call)
        else:
            self.queues[call.rank].append(call)
        return e

    def assign_waiting(self, e: Employee) -> None:
        for rank in reversed(Rank):  # serve the highest-priority queue this employee can handle
            if rank <= e.rank and self.queues[rank]:
                e.receive(self.queues[rank].popleft())
                return


cc = CallCenter(respondents=2, managers=1, directors=1)
calls = [Call(f"caller{i}") for i in range(5)]
handlers = [cc.dispatch_call(c) for c in calls]
assert [h.name if h else None for h in handlers] == ["R0", "R1", "M0", "D0", None]
calls[0].handler.escalate()  # R0 can't handle caller0 -> queued for a manager; R0 picks up caller4
assert calls[4].handler.name == "R0" and cc.queues[Rank.MANAGER][0] is calls[0]
calls[2].handler.complete()  # M0 frees up and takes the escalated call
assert calls[0].handler.name == "M0"
print("7.2 ok")

7.2 ok


## 7.3 Jukebox
A jukebox holds CDs of songs, a playlist queue, and a current user. It can play, skip, and queue songs.

In [4]:
@dataclass(frozen=True)
class Song:
    title: str
    artist: str
    seconds: int


@dataclass
class CD:
    title: str
    songs: list[Song]


class Playlist:
    def __init__(self) -> None:
        self._queue: deque[Song] = deque()

    def enqueue(self, s: Song) -> None: self._queue.append(s)
    def next(self) -> Song | None: return self._queue.popleft() if self._queue else None
    def __len__(self) -> int: return len(self._queue)


class Jukebox:
    def __init__(self, cds: list[CD]) -> None:
        self.cds = cds
        self.playlist = Playlist()
        self.now_playing: Song | None = None
        self.credits = 0

    def insert_coin(self, n: int = 1) -> None:
        self.credits += n

    def search(self, text: str) -> list[Song]:
        text = text.lower()
        return [s for cd in self.cds for s in cd.songs if text in s.title.lower() or text in s.artist.lower()]

    def request(self, song: Song) -> None:
        if self.credits <= 0:
            raise PermissionError("insert coin")
        self.credits -= 1
        self.playlist.enqueue(song)
        if self.now_playing is None:
            self.skip()

    def skip(self) -> Song | None:
        self.now_playing = self.playlist.next()
        return self.now_playing


jb = Jukebox([CD("Hits", [Song("Blue Sky", "Band A", 200), Song("Red Sun", "Band B", 180)])])
jb.insert_coin(2)
jb.request(jb.search("blue")[0]); jb.request(jb.search("band b")[0])
assert jb.now_playing.title == "Blue Sky" and jb.skip().title == "Red Sun" and jb.skip() is None
print("7.3 ok")

7.3 ok


## 7.4 Parking Lot
Multiple levels, each with rows of spots of different sizes (motorcycle, compact, large). A motorcycle fits any spot, a car fits compact or large, and a bus needs **5 consecutive large spots in the same row**.

In [5]:
class Size(IntEnum):
    MOTORCYCLE = 0
    COMPACT = 1
    LARGE = 2


class Vehicle(ABC):
    size: ClassVar[Size]
    spots_needed: ClassVar[int] = 1

    def __init__(self, plate: str) -> None:
        self.plate = plate
        self.spots: list[ParkingSpot] = []

    def can_fit(self, spot: ParkingSpot) -> bool:
        return spot.size >= self.size


class Motorcycle(Vehicle): size = Size.MOTORCYCLE
class Car(Vehicle): size = Size.COMPACT
class Bus(Vehicle):
    size = Size.LARGE
    spots_needed = 5


@dataclass(eq=False)
class ParkingSpot:
    level: int
    row: int
    number: int
    size: Size
    vehicle: Vehicle | None = None


class Level:
    def __init__(self, floor: int, rows: list[list[Size]]) -> None:
        self.floor = floor
        self.rows = [[ParkingSpot(floor, r, i, s) for i, s in enumerate(row)] for r, row in enumerate(rows)]

    def park(self, v: Vehicle) -> bool:
        for row in self.rows:
            run: list[ParkingSpot] = []
            for spot in row:
                if spot.vehicle is None and v.can_fit(spot):
                    run.append(spot)
                    if len(run) == v.spots_needed:
                        for s in run:
                            s.vehicle = v
                        v.spots = run
                        return True
                else:
                    run = []
        return False

    def free_spots(self) -> int:
        return sum(s.vehicle is None for row in self.rows for s in row)


class ParkingLot:
    def __init__(self, levels: list[Level]) -> None:
        self.levels = levels

    def park(self, v: Vehicle) -> bool:
        return any(level.park(v) for level in self.levels)

    def leave(self, v: Vehicle) -> None:
        for s in v.spots:
            s.vehicle = None
        v.spots = []


row = [Size.MOTORCYCLE] * 2 + [Size.COMPACT] * 3 + [Size.LARGE] * 5
lot = ParkingLot([Level(0, [row]), Level(1, [row])])
bus, car, moto = Bus("BUS1"), Car("CAR1"), Motorcycle("M1")
assert lot.park(bus) and all(s.size == Size.LARGE for s in bus.spots)
assert lot.park(car) and lot.park(moto)
assert lot.park(Bus("BUS2")) and bus.spots[0].level == 0
assert not lot.park(Bus("BUS3"))
lot.leave(bus)
assert lot.park(Bus("BUS3"))
print("7.4 ok")

7.4 ok


## 7.5 Online Book Reader
Users (with memberships), a library of books, and a reader that tracks each user's active book and page.

In [6]:
@dataclass
class Book:
    id: int
    title: str
    pages: list[str]


@dataclass
class User:
    id: int
    name: str
    membership_expires: datetime


class Library:
    def __init__(self) -> None:
        self._books: dict[int, Book] = {}

    def add(self, b: Book) -> None: self._books[b.id] = b
    def remove(self, book_id: int) -> None: self._books.pop(book_id, None)
    def find(self, book_id: int) -> Book | None: return self._books.get(book_id)


class UserManager:
    def __init__(self) -> None:
        self._users: dict[int, User] = {}

    def add(self, u: User) -> None: self._users[u.id] = u
    def find(self, user_id: int) -> User | None: return self._users.get(user_id)


class OnlineReader:
    def __init__(self) -> None:
        self.library = Library()
        self.users = UserManager()
        self._sessions: dict[int, tuple[Book, int]] = {}  # user -> (book, page index)

    def open(self, user_id: int, book_id: int) -> str:
        user, book = self.users.find(user_id), self.library.find(book_id)
        if not user or not book:
            raise KeyError("unknown user or book")
        if user.membership_expires < datetime.now():
            raise PermissionError("membership expired")
        self._sessions[user_id] = (book, 0)
        return book.pages[0]

    def turn(self, user_id: int, delta: int = 1) -> str:
        book, page = self._sessions[user_id]
        page = max(0, min(len(book.pages) - 1, page + delta))
        self._sessions[user_id] = (book, page)
        return book.pages[page]


reader = OnlineReader()
reader.library.add(Book(1, "Py", ["p1", "p2", "p3"]))
reader.users.add(User(7, "ann", datetime.now() + timedelta(days=30)))
assert reader.open(7, 1) == "p1" and reader.turn(7) == "p2" and reader.turn(7, 5) == "p3"
print("7.5 ok")

7.5 ok


## 7.6 Jigsaw
An N×N jigsaw puzzle. Design the data structures and a `solve` algorithm, given a `fits_with(edge_a, edge_b)` primitive. Pieces have 4 edges, each `FLAT`, `INNER`, or `OUTER`. We generate a random puzzle, shuffle and rotate the pieces, then solve it by placing corners, edges, and interior pieces one at a time with rotation.

In [7]:
class Shape_(Enum):
    FLAT = 0; INNER = 1; OUTER = 2


@dataclass
class Edge:
    shape: Shape_
    code: int  # a matching pair shares a code; INNER fits OUTER with the same code

    def fits_with(self, other: Edge) -> bool:
        return {self.shape, other.shape} == {Shape_.INNER, Shape_.OUTER} and self.code == other.code


@dataclass(eq=False)
class Piece:
    edges: list[Edge]  # order: top, right, bottom, left
    id: int = 0

    def rotate(self) -> None:  # 90 degrees clockwise
        self.edges = self.edges[-1:] + self.edges[:-1]

    def flat_count(self) -> int:
        return sum(e.shape is Shape_.FLAT for e in self.edges)


def make_puzzle(n: int) -> list[Piece]:
    codes = iter(range(10**6))
    horiz = [[next(codes) for _ in range(n)] for _ in range(n - 1)]  # between rows
    vert = [[next(codes) for _ in range(n - 1)] for _ in range(n)]   # between cols
    pieces = []
    for r in range(n):
        for c in range(n):
            def e(code: int | None, outer: bool) -> Edge:
                if code is None: return Edge(Shape_.FLAT, -1)
                return Edge(Shape_.OUTER if outer else Shape_.INNER, code)
            top = e(horiz[r - 1][c] if r > 0 else None, False)
            bottom = e(horiz[r][c] if r < n - 1 else None, True)
            left = e(vert[r][c - 1] if c > 0 else None, False)
            right = e(vert[r][c] if c < n - 1 else None, True)
            pieces.append(Piece([top, right, bottom, left], id=r * n + c))
    random.shuffle(pieces)
    for p in pieces:
        for _ in range(random.randrange(4)):
            p.rotate()
    return pieces


def solve_jigsaw(pieces: list[Piece], n: int) -> list[list[Piece]]:
    grid: list[list[Piece | None]] = [[None] * n for _ in range(n)]
    remaining = set(pieces)
    for r in range(n):
        for c in range(n):
            need_flat = [r == 0, c == n - 1, r == n - 1, c == 0]
            for p in list(remaining):
                for _ in range(4):
                    ok = all((p.edges[i].shape is Shape_.FLAT) == need_flat[i] for i in range(4))
                    if ok and r > 0:
                        ok = p.edges[0].fits_with(grid[r - 1][c].edges[2])
                    if ok and c > 0:
                        ok = p.edges[3].fits_with(grid[r][c - 1].edges[1])
                    if ok:
                        break
                    p.rotate()
                if ok:
                    grid[r][c] = p
                    remaining.discard(p)
                    break
            else:
                raise RuntimeError(f"no piece fits at {(r, c)}")
    return grid  # type: ignore[return-value]


N = 5
solved = solve_jigsaw(make_puzzle(N), N)
# Any of the 4 corners can start the solve, so the answer may be the original picture rotated.
# Instead of comparing ids, check that every neighbouring pair of edges fits.
assert all(solved[r][c].edges[1].fits_with(solved[r][c + 1].edges[3]) for r in range(N) for c in range(N - 1))
assert all(solved[r][c].edges[2].fits_with(solved[r + 1][c].edges[0]) for r in range(N - 1) for c in range(N))
print("7.6 ok\n" + "\n".join(" ".join(f"{p.id:2}" for p in row) for row in solved))

7.6 ok
24 23 22 21 20
19 18 17 16 15
14 13 12 11 10
 9  8  7  6  5
 4  3  2  1  0


## 7.7 Chat Server
Core objects: users, contact requests, private/group conversations, and messages with timestamps. The networking layer is out of scope; the server is an in-memory coordinator.

In [8]:
class RequestStatus(Enum):
    PENDING = auto(); ACCEPTED = auto(); REJECTED = auto()


@dataclass
class Message:
    sender: str
    content: str
    at: datetime = field(default_factory=datetime.now)


@dataclass(eq=False)
class Conversation:
    id: int
    participants: set[str]
    messages: list[Message] = field(default_factory=list)


@dataclass(eq=False)
class ChatUser:
    name: str
    contacts: set[str] = field(default_factory=set)
    inbox: list[tuple[str, RequestStatus]] = field(default_factory=list)
    conversations: dict[int, Conversation] = field(default_factory=dict)
    online: bool = False


class ChatServer:
    def __init__(self) -> None:
        self.users: dict[str, ChatUser] = {}
        self._next_conv = 0

    def sign_up(self, name: str) -> ChatUser:
        self.users[name] = u = ChatUser(name)
        return u

    def send_request(self, frm: str, to: str) -> None:
        self.users[to].inbox.append((frm, RequestStatus.PENDING))

    def respond(self, user: str, frm: str, accept: bool) -> None:
        u = self.users[user]
        u.inbox = [(f, s) for f, s in u.inbox if f != frm]
        if accept:
            u.contacts.add(frm)
            self.users[frm].contacts.add(user)

    def start_conversation(self, *names: str) -> Conversation:
        starter = self.users[names[0]]
        if not all(n in starter.contacts for n in names[1:]):
            raise PermissionError("can only chat with contacts")
        conv = Conversation(self._next_conv, set(names))
        self._next_conv += 1
        for n in names:
            self.users[n].conversations[conv.id] = conv
        return conv

    def send(self, sender: str, conv_id: int, content: str) -> None:
        conv = self.users[sender].conversations[conv_id]
        conv.messages.append(Message(sender, content))


server = ChatServer()
for n in ("ann", "bob", "cat"):
    server.sign_up(n)
server.send_request("ann", "bob"); server.respond("bob", "ann", accept=True)
conv = server.start_conversation("ann", "bob")
server.send("bob", conv.id, "hi ann")
assert server.users["ann"].conversations[conv.id].messages[0].content == "hi ann"
try:
    server.start_conversation("ann", "cat"); raise AssertionError
except PermissionError:
    pass
print("7.7 ok")

7.7 ok


## 7.8 Othello
An 8×8 board. A move must flip at least one line of opponent pieces, and the game ends when neither player can move.

In [9]:
class Color(Enum):
    BLACK = "B"; WHITE = "W"

    @property
    def other(self) -> Color:
        return Color.WHITE if self is Color.BLACK else Color.BLACK


DIRS = [(dr, dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if (dr, dc) != (0, 0)]


class Othello:
    def __init__(self, size: int = 8) -> None:
        self.n = size
        self.board: list[list[Color | None]] = [[None] * size for _ in range(size)]
        m = size // 2
        self.board[m - 1][m - 1] = self.board[m][m] = Color.WHITE
        self.board[m - 1][m] = self.board[m][m - 1] = Color.BLACK
        self.turn = Color.BLACK

    def _flips(self, r: int, c: int, color: Color) -> list[tuple[int, int]]:
        if self.board[r][c] is not None:
            return []
        flips = []
        for dr, dc in DIRS:
            line, rr, cc = [], r + dr, c + dc
            while 0 <= rr < self.n and 0 <= cc < self.n and self.board[rr][cc] is color.other:
                line.append((rr, cc)); rr += dr; cc += dc
            if line and 0 <= rr < self.n and 0 <= cc < self.n and self.board[rr][cc] is color:
                flips += line
        return flips

    def legal_moves(self, color: Color) -> list[tuple[int, int]]:
        return [(r, c) for r in range(self.n) for c in range(self.n) if self._flips(r, c, color)]

    def play(self, r: int, c: int) -> None:
        flips = self._flips(r, c, self.turn)
        if not flips:
            raise ValueError("illegal move")
        for rr, cc in flips + [(r, c)]:
            self.board[rr][cc] = self.turn
        if self.legal_moves(self.turn.other):
            self.turn = self.turn.other

    def game_over(self) -> bool:
        return not self.legal_moves(Color.BLACK) and not self.legal_moves(Color.WHITE)

    def score(self) -> dict[Color, int]:
        return {col: sum(cell is col for row in self.board for cell in row) for col in Color}


game = Othello()
assert sorted(game.legal_moves(Color.BLACK)) == [(2, 3), (3, 2), (4, 5), (5, 4)]
while not game.game_over():
    game.play(*random.choice(game.legal_moves(game.turn)))
print("final score:", {c.name: s for c, s in game.score().items()})

final score: {'BLACK': 38, 'WHITE': 26}


## 7.9 Circular Array
A `CircularArray` class supporting efficient rotation, and iterable with `for x in arr`. Rotation just moves a `head` offset: `O(1)`.

In [10]:
class CircularArray(Generic[T]):
    def __init__(self, items: list[T]) -> None:
        self._items = list(items)
        self._head = 0

    def _idx(self, i: int) -> int:
        if not -len(self) <= i < len(self):
            raise IndexError(i)
        return (self._head + i) % len(self._items)

    def rotate(self, shift_right: int) -> None:
        self._head = (self._head + shift_right) % len(self._items)

    def __getitem__(self, i: int) -> T: return self._items[self._idx(i)]
    def __setitem__(self, i: int, v: T) -> None: self._items[self._idx(i)] = v
    def __len__(self) -> int: return len(self._items)

    def __iter__(self) -> Iterator[T]:
        for i in range(len(self)):
            yield self[i]


ca = CircularArray([1, 2, 3, 4, 5])
ca.rotate(2)
assert list(ca) == [3, 4, 5, 1, 2] and ca[-1] == 2
print("7.9 ok")

7.9 ok


## 7.10 Minesweeper
N×N board with B hidden bombs. Clicking a bomb loses. Clicking a numbered cell reveals it; clicking a blank cell (zero adjacent bombs) flood-fills outward. Win when all non-bomb cells are revealed.

In [11]:
class GameState(Enum):
    PLAYING = auto(); WON = auto(); LOST = auto()


@dataclass
class Cell:
    bomb: bool = False
    number: int = 0
    revealed: bool = False
    flagged: bool = False


class Minesweeper:
    def __init__(self, n: int, bombs: int, seed: int | None = None) -> None:
        self.n = n
        rng = random.Random(seed)
        self.cells = [[Cell() for _ in range(n)] for _ in range(n)]
        for pos in rng.sample(range(n * n), bombs):  # "shuffle" bomb positions
            self.cells[pos // n][pos % n].bomb = True
        for r in range(n):
            for c in range(n):
                self.cells[r][c].number = sum(self.cells[rr][cc].bomb for rr, cc in self._neighbours(r, c))
        self.hidden_safe = n * n - bombs
        self.state = GameState.PLAYING

    def _neighbours(self, r: int, c: int) -> Iterator[tuple[int, int]]:
        for dr, dc in DIRS:
            if 0 <= r + dr < self.n and 0 <= c + dc < self.n:
                yield r + dr, c + dc

    def click(self, r: int, c: int) -> GameState:
        cell = self.cells[r][c]
        if cell.revealed or cell.flagged:
            return self.state
        if cell.bomb:
            self.state = GameState.LOST
            return self.state
        q = deque([(r, c)])  # BFS flood fill of blanks
        cell.revealed = True
        self.hidden_safe -= 1
        while q:
            cr, cc = q.popleft()
            if self.cells[cr][cc].number == 0:
                for nr, nc in self._neighbours(cr, cc):
                    nb = self.cells[nr][nc]
                    if not nb.revealed and not nb.bomb:
                        nb.revealed = True
                        self.hidden_safe -= 1
                        q.append((nr, nc))
        if self.hidden_safe == 0:
            self.state = GameState.WON
        return self.state

    def __str__(self) -> str:
        def ch(x: Cell) -> str:
            if not x.revealed: return "?"
            return str(x.number) if x.number else " "
        return "\n".join("".join(ch(x) for x in row) for row in self.cells)


ms = Minesweeper(8, 10, seed=1)
safe = [(r, c) for r in range(8) for c in range(8) if not ms.cells[r][c].bomb]
for r, c in safe:
    ms.click(r, c)
assert ms.state is GameState.WON
ms2 = Minesweeper(8, 10, seed=1)
blank = next((r, c) for r, c in safe if ms2.cells[r][c].number == 0)
ms2.click(*blank)
print(ms2)

   1????
2211????
????????
????????
????????
????????
????????
????????


## 7.11 File System
An in-memory file system with directories and files, both kinds of entry.

In [12]:
class Entry(ABC):
    def __init__(self, name: str, parent: Directory | None) -> None:
        self.name = name
        self.parent = parent
        self.created = self.modified = datetime.now()

    @abstractmethod
    def size(self) -> int: ...

    def path(self) -> str:
        if self.parent is None:
            return "/"
        base = self.parent.path()
        return f"{base.rstrip('/')}/{self.name}"

    def delete(self) -> None:
        if self.parent:
            self.parent.children.pop(self.name)


class File(Entry):
    def __init__(self, name: str, parent: Directory, content: bytes = b"") -> None:
        super().__init__(name, parent)
        self.content = content

    def size(self) -> int:
        return len(self.content)


class Directory(Entry):
    def __init__(self, name: str = "", parent: Directory | None = None) -> None:
        super().__init__(name, parent)
        self.children: dict[str, Entry] = {}

    def size(self) -> int:
        return sum(c.size() for c in self.children.values())

    def mkdir(self, name: str) -> Directory:
        d = Directory(name, self)
        self.children[name] = d
        return d

    def touch(self, name: str, content: bytes = b"") -> File:
        f = File(name, self, content)
        self.children[name] = f
        return f

    def resolve(self, path: str) -> Entry:
        node: Entry = self
        for part in filter(None, path.split("/")):
            if not isinstance(node, Directory) or part not in node.children:
                raise FileNotFoundError(path)
            node = node.children[part]
        return node


root_dir = Directory()
docs = root_dir.mkdir("home").mkdir("docs")
docs.touch("a.txt", b"hello")
docs.touch("b.txt", b"world!")
assert root_dir.size() == 11 and root_dir.resolve("/home/docs/b.txt").path() == "/home/docs/b.txt"
docs.children["a.txt"].delete()
assert root_dir.size() == 6
print("7.11 ok")

7.11 ok


## 7.12 Hash Table
Hash table with chaining via linked lists (Chapter 1 used Python lists as buckets; here each bucket is a linked list, as in the book).

In [13]:
@dataclass
class LLNode(Generic[T]):
    key: object
    value: T
    next: LLNode[T] | None = None


class ChainedHashTable(Generic[T]):
    def __init__(self, capacity: int = 16) -> None:
        self._buckets: list[LLNode[T] | None] = [None] * capacity

    def _i(self, key: object) -> int:
        return hash(key) % len(self._buckets)

    def put(self, key: object, value: T) -> None:
        i = self._i(key)
        node = self._buckets[i]
        while node:
            if node.key == key:
                node.value = value
                return
            node = node.next
        self._buckets[i] = LLNode(key, value, self._buckets[i])

    def get(self, key: object) -> T | None:
        node = self._buckets[self._i(key)]
        while node:
            if node.key == key:
                return node.value
            node = node.next
        return None

    def remove(self, key: object) -> None:
        i = self._i(key)
        prev, node = None, self._buckets[i]
        while node:
            if node.key == key:
                if prev: prev.next = node.next
                else: self._buckets[i] = node.next
                return
            prev, node = node, node.next


cht: ChainedHashTable[int] = ChainedHashTable(capacity=2)  # force collisions
for i in range(10):
    cht.put(f"k{i}", i)
cht.remove("k3")
assert cht.get("k7") == 7 and cht.get("k3") is None
print("7.12 ok")

7.12 ok
